# Four models on the same PlantVillage dataset

Compare the original repository CNN, MobileNetV2, EfficientNetB0, and DenseNet121 on all 38 classes from `raw/color`. The saved split manifest contains identical 70% train / 15% validation / 15% test membership and image hashes for every model.

The original Conv32/Pool3/Conv16/Pool2/Flatten/Dense8 network trains from scratch with 38 outputs. The other three use frozen ImageNet encoders and a Dense256/Dropout0.3 head. All use 128 × 128 RGB inputs, class weights, Adam(0.001), and up to 30 epochs. The original notebook used 256-pixel images; the common resolution makes this experiment practical on CPU. This compares training approaches, with one seed, rather than full fine-tuning at native model resolutions.


In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

output = Path("comparison")


## Run or resume the comparison
Completed compatible runs and extracted features are reused. Downloads are needed on the first run for pretrained weights. This cell may take substantial CPU time.


In [ ]:
subprocess.run([sys.executable, "compare_models.py", "--epochs", "30"], check=True)


## Overall results
The preferred model is chosen by validation macro F1. Test metrics are for reporting and do not determine model selection.


In [ ]:
report = json.loads((output / "comparison_results.json").read_text())
print("Preferred by validation macro F1:", report["preferred_model_by_validation_macro_f1"])
results = pd.read_csv(output / "comparison_results.csv")
results


In [ ]:
plt.figure(figsize=(15, 5))
plt.imshow(plt.imread(output / "comparison_chart.png"))
plt.axis("off")
plt.show()


## Results for each class
Macro F1 gives each class equal weight. The per-class table makes weaker disease categories visible even when overall accuracy is high.


In [ ]:
class_f1 = {}
for entry in report["results"]:
    metrics = json.loads((output / entry["model"] / "test_class_metrics.json").read_text())
    class_f1[entry["model"]] = {name: scores["f1"] for name, scores in metrics.items()}
pd.DataFrame(class_f1).style.format("{:.1%}").background_gradient(cmap="YlGn", vmin=0, vmax=1)


## Confusion matrices
Each row is normalized by its actual class count, so minority classes remain visible. Class indices follow the saved class mapping.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
for axis, entry in zip(axes.flat, report["results"]):
    matrix = np.load(output / entry["model"] / "confusion_matrix.npy")
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    image = axis.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
    axis.set_title(entry["model"])
    axis.set_xlabel("Predicted class index")
    axis.set_ylabel("Actual class index")
fig.colorbar(image, ax=axes.ravel().tolist(), label="Fraction of actual class")
plt.show()
pd.DataFrame({"Class index": range(len(report["protocol"]["class_names"])),
              "Class": report["protocol"]["class_names"]})


## Training curves
All best checkpoints are selected using validation loss; early stopping may produce different epoch counts within the same maximum budget.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for entry in report["results"]:
    history = json.loads((output / entry["model"] / "training_history.json").read_text())
    axes[0].plot(range(1, len(history["val_accuracy"]) + 1), history["val_accuracy"], label=entry["model"])
    axes[1].plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], label=entry["model"])
for axis, label in zip(axes, ["Validation accuracy", "Validation loss"]):
    axis.set_xlabel("Epoch")
    axis.set_ylabel(label)
    axis.legend()
plt.tight_layout()
